# Phase 2 — Classical ML
## Day 10: SHAP & Mini Project
**Date:** 2026-04-24

### What you'll learn today
- What SHAP values are and why they matter
- SHAP summary plots and force plots
- Feature importance: model-based vs SHAP
- Mini project: Credit Risk Scoring with full SHAP explainability
- Mini project: Customer Segmentation (KMeans + SHAP on a downstream classifier)

In [ ]:
# Setup — install shap if needed
# pip install shap xgboost scikit-learn pandas numpy matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.cluster import KMeans
import xgboost as xgb
import shap

print("All imports OK")
print(f"SHAP version: {shap.__version__}")

In [ ]:
# --- Synthetic Credit Risk Dataset ---
np.random.seed(42)
n = 1000

age           = np.random.randint(22, 65, n)
income        = np.random.normal(50000, 15000, n).clip(15000, 120000)
loan_amount   = np.random.normal(12000, 5000, n).clip(2000, 50000)
credit_score  = np.random.randint(300, 850, n)
num_defaults  = np.random.poisson(0.5, n)
employment_yrs= np.random.randint(0, 30, n)
debt_ratio    = loan_amount / income

# Build target: higher score & income = lower risk
log_odds = (
    -3
    + 0.02  * (700 - credit_score)   # low score → more risk
    + 0.00002 * (50000 - income)      # low income → more risk
    + 0.5   * num_defaults
    - 0.03  * employment_yrs
    + 2     * debt_ratio
)
prob_default = 1 / (1 + np.exp(-log_odds))
default = (np.random.random(n) < prob_default).astype(int)

df = pd.DataFrame({
    'age': age,
    'income': income,
    'loan_amount': loan_amount,
    'credit_score': credit_score,
    'num_defaults': num_defaults,
    'employment_yrs': employment_yrs,
    'debt_ratio': debt_ratio,
    'default': default
})

print(df.shape)
print(df['default'].value_counts())
df.head()

## 1. Model-Based Feature Importance

Tree models like Random Forest and XGBoost have a built-in `.feature_importances_` property.
It measures how much each feature reduces impurity (Gini) across all splits in all trees.

It's fast and easy, but it has a big weakness: it tends to **favor high-cardinality features**
(features with many unique values like income or age) even if they're not actually that useful.
It also tells you *global* importance only — not why a *specific* prediction was made.

That's where SHAP comes in.

In [ ]:
# Train a Random Forest
FEATURES = ['age', 'income', 'loan_amount', 'credit_score',
            'num_defaults', 'employment_yrs', 'debt_ratio']

X = df[FEATURES]
y = df['default']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Plot model-based feature importance
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)
importances.plot(kind='barh', figsize=(7, 4), title='RF Feature Importances (Gini)')
plt.tight_layout()
plt.show()
print(importances.sort_values(ascending=False))

## 2. What Is SHAP?

SHAP (SHapley Additive exPlanations) comes from game theory. The idea is simple:
imagine each feature as a "player" in a game. SHAP figures out each player's **fair contribution**
to the final prediction by trying every possible combination of features.

For a single prediction, the SHAP values sum up like this:

> `prediction = base_value + SHAP(feature_1) + SHAP(feature_2) + ...`

The `base_value` is just the average prediction across the whole training set.
Each SHAP value is positive (pushes toward default) or negative (pushes away from default).

### Why it beats plain feature importance
- It works per-prediction, not just globally
- It handles feature interactions properly
- It's consistent: a feature that matters more *always* gets a higher SHAP value
- It works on any model (not just trees)

In [ ]:
# Compute SHAP values with TreeExplainer (fast for tree models)
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

# For binary classification, shap_values is a list of 2 arrays [class0, class1]
# We want class 1 (default)
shap_vals_default = shap_values[1]  # shape: (n_test_samples, n_features)

print(f"SHAP values shape: {shap_vals_default.shape}")
print(f"Base value (avg prediction): {explainer.expected_value[1]:.3f}")
print(f"First test sample SHAP sum: {shap_vals_default[0].sum():.3f}")
print(f"Model prediction for first sample: {rf.predict_proba(X_test.iloc[[0]])[0][1]:.3f}")

## 3. SHAP Summary Plot

The summary plot combines feature importance with feature effects in one chart.
Each dot is one test sample. The x-axis shows the SHAP value (impact on prediction).
Color shows the feature value: red = high, blue = low.

Read it like this:
- Features at the top matter most (highest average |SHAP|)
- Red dots on the right = high feature value pushes toward default
- Blue dots on the right = low feature value pushes toward default

In [ ]:
# SHAP Summary Plot — beeswarm style (default)
shap.summary_plot(shap_vals_default, X_test, feature_names=FEATURES, show=True)

In [ ]:
# SHAP Summary Plot — bar style (just global importance)
shap.summary_plot(shap_vals_default, X_test, feature_names=FEATURES,
                  plot_type='bar', show=True)

## 4. SHAP Force Plot (Single Prediction)

A force plot shows *why* the model made a specific prediction for one person.
The base value is the average prediction. Red features push the score up (toward default).
Blue features push it down (away from default). The final prediction is where the forces balance.

This is how you explain "why was this person flagged?" to a business stakeholder.

In [ ]:
# Pick a high-risk person from the test set
probs = rf.predict_proba(X_test)[:, 1]
high_risk_idx = probs.argmax()

print(f"Highest risk sample (index {high_risk_idx}):")
print(X_test.iloc[high_risk_idx])
print(f"Predicted default probability: {probs[high_risk_idx]:.2%}")

# Force plot (saved as matplotlib figure so it works in Jupyter without JS)
shap.initjs()
force_plot = shap.force_plot(
    explainer.expected_value[1],
    shap_vals_default[high_risk_idx],
    X_test.iloc[high_risk_idx],
    feature_names=FEATURES,
    matplotlib=True,
    show=True
)

## 5. SHAP Waterfall Plot

The waterfall plot is another way to show a single prediction.
It starts from the base value and stacks each feature's contribution step by step.
Red bars = features that increase risk. Blue bars = features that reduce risk.

In [ ]:
# Waterfall plot for same high-risk sample
explanation = shap.Explanation(
    values=shap_vals_default[high_risk_idx],
    base_values=explainer.expected_value[1],
    data=X_test.iloc[high_risk_idx].values,
    feature_names=FEATURES
)
shap.waterfall_plot(explanation, show=True)

## 6. SHAP with XGBoost

XGBoost has native SHAP support built in (even faster than the generic TreeExplainer).
Let's compare feature rankings between RF and XGBoost.

In [ ]:
# Train XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=4,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)
xgb_model.fit(X_train, y_train)

# SHAP values via TreeExplainer
xgb_explainer = shap.TreeExplainer(xgb_model)
xgb_shap_vals = xgb_explainer.shap_values(X_test)

print("XGBoost test AUC:", roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:, 1]).round(3))
print("Random Forest test AUC:", roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]).round(3))

In [ ]:
# Compare SHAP importance: RF vs XGBoost
rf_mean_shap  = np.abs(shap_vals_default).mean(axis=0)
xgb_mean_shap = np.abs(xgb_shap_vals).mean(axis=0)

compare_df = pd.DataFrame({
    'Feature': FEATURES,
    'RF_SHAP': rf_mean_shap,
    'XGB_SHAP': xgb_mean_shap
}).set_index('Feature').sort_values('RF_SHAP', ascending=False)

print(compare_df.round(4))
compare_df.plot(kind='barh', figsize=(8, 4), title='Mean |SHAP| — RF vs XGBoost')
plt.tight_layout()
plt.show()

## 7. Mini Project Part 1 — Customer Segmentation

We'll cluster customers into groups using KMeans. Then we'll train a classifier to *predict*
which cluster a customer belongs to, and use SHAP to understand what defines each cluster.
This is a common trick to make unsupervised clusters interpretable.

In [ ]:
# Cluster customers into 3 segments based on financial features
seg_features = ['income', 'credit_score', 'debt_ratio', 'num_defaults', 'employment_yrs']
X_seg = df[seg_features].copy()

# Scale before clustering
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_seg)

# KMeans clustering
km = KMeans(n_clusters=3, random_state=42, n_init=10)
df['segment'] = km.fit_predict(X_scaled)

print("Cluster sizes:")
print(df['segment'].value_counts().sort_index())

# Profile each cluster
print("\nCluster profiles:")
print(df.groupby('segment')[seg_features].mean().round(1))

In [ ]:
# Train a classifier to predict segment labels — then explain with SHAP
X_seg_full = df[seg_features]
y_seg = df['segment']

X_str, X_ste, y_str, y_ste = train_test_split(X_seg_full, y_seg, test_size=0.2, random_state=42)

seg_clf = RandomForestClassifier(n_estimators=100, random_state=42)
seg_clf.fit(X_str, y_str)
print("Segment classification accuracy:", seg_clf.score(X_ste, y_ste).round(3))

# SHAP for segment classifier (multiclass — one array per class)
seg_explainer = shap.TreeExplainer(seg_clf)
seg_shap_vals = seg_explainer.shap_values(X_ste)  # list of 3 arrays

print(f"\nSHAP arrays (one per segment): {len(seg_shap_vals)}")
print(f"Each array shape: {seg_shap_vals[0].shape}")

In [ ]:
# SHAP summary for each segment
segment_names = {0: 'Segment 0', 1: 'Segment 1', 2: 'Segment 2'}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i in range(3):
    plt.sca(axes[i])
    shap.summary_plot(
        seg_shap_vals[i], X_ste,
        feature_names=seg_features,
        plot_type='bar',
        show=False,
        ax=axes[i]
    )
    axes[i].set_title(f'What defines {segment_names[i]}?')
plt.tight_layout()
plt.show()

## Tricky Bits — Common Mistakes

In [ ]:
# Mistake 1: Forgetting that shap_values is a LIST for multiclass
try:
    rf_single_class = RandomForestClassifier(n_estimators=10, random_state=0)
    rf_single_class.fit(X_train, y_train)
    exp = shap.TreeExplainer(rf_single_class)
    sv = exp.shap_values(X_test)

    # Binary RF returns a list of 2 arrays — always index with [1] for class 1
    print(f"Type: {type(sv)}, len: {len(sv)}")
    print(f"Correct — sv[1] shape: {sv[1].shape}")
    print(f"Wrong — sv.shape would fail...")
    sv.shape  # This will crash
except AttributeError as e:
    print(f"AttributeError caught: {e}")
    print("Fix: use sv[1] not sv for binary classification")

In [ ]:
# Mistake 2: Interpreting SHAP values as probabilities
# SHAP values are in log-odds space for classifiers — not probabilities!
sample_shap = shap_vals_default[0]
base = explainer.expected_value[1]

print(f"Base value: {base:.3f}")
print(f"Sum of SHAPs: {sample_shap.sum():.3f}")
print(f"Base + SHAPs: {base + sample_shap.sum():.3f}")
print(f"Model predict_proba: {rf.predict_proba(X_test.iloc[[0]])[0][1]:.3f}")
print()
print("Note: for RF, SHAP values are in probability space (not log-odds)")
print("For XGBoost/LogReg, they're in log-odds space — you'd need sigmoid to get probability")

In [ ]:
# Mistake 3: Running SHAP on the whole dataset (very slow)
# Always compute on a sample or just X_test
import time

# Fast: X_test (200 samples)
t0 = time.time()
_ = explainer.shap_values(X_test)
print(f"X_test (200 rows): {time.time()-t0:.2f}s")

# Don't do this on huge datasets without sampling
print("Tip: for big datasets, use shap.sample(X, 100) as background data")

## Trick Questions

**Q1: SHAP values for a sample always sum to exactly the model prediction. True or false?**

<details><summary>Answer</summary>

True — that's the key property of SHAP. `base_value + sum(shap_values) = model_output`.
The "model_output" is in the same space as the model's raw output (log-odds for XGBoost, probability for RF).

</details>

---

**Q2: If a feature has zero SHAP value for a prediction, does that mean it was not used by the model?**

<details><summary>Answer</summary>

No. It means that feature had no *marginal* effect on *this specific prediction* — but the model still used it.
It might have contributed in one split and been cancelled out in another, or the value happened to be at the average.

</details>

---

**Q3: Can SHAP values be negative?**

<details><summary>Answer</summary>

Yes. A negative SHAP value means that feature *reduced* the predicted probability compared to the baseline.
For credit risk: a high credit score would have a large negative SHAP value (it pushes *away* from default).

</details>

---

**Q4: TreeExplainer is exact. KernelExplainer is an approximation. Which is faster?**

<details><summary>Answer</summary>

TreeExplainer (exact) is much faster because it exploits the tree structure.
KernelExplainer (approximation) is model-agnostic and works on any model but is very slow for large datasets.
Use TreeExplainer for any tree-based model.

</details>

---

**Q5: You train an XGBoost model and RF on the same data. XGBoost has higher AUC but RF SHAP says `credit_score` is most important while XGBoost SHAP says `debt_ratio` is. Which is right?**

<details><summary>Answer</summary>

Both are right — for their respective models. Different models can learn different interactions.
SHAP explains *the model's* reasoning, not ground truth. If the models disagree, investigate whether there's a data issue or if one model has overfitted.

</details>

## Exercises

Fill in the `___` blanks. `assert` will confirm if you're right.

In [ ]:
# Exercise 1: Compute SHAP values for XGBoost on X_test
# and find the feature with the highest mean absolute SHAP value

xgb_shap_abs_mean = np.abs(___).mean(axis=0)   # hint: use xgb_shap_vals
most_important_feature = FEATURES[___.argmax()]  # hint: use xgb_shap_abs_mean

print(f"Most important feature (XGB SHAP): {most_important_feature}")
assert most_important_feature in FEATURES, "Must be one of the feature names"
print("Pass!")

In [ ]:
# Exercise 2: Find the single test sample with the LARGEST total SHAP impact
# (sum of absolute SHAP values across all features — most "explained" sample)

shap_total_impact = np.abs(shap_vals_default).sum(axis=___)  # sum across features
most_explained_idx = shap_total_impact.___()  # index of the max

print(f"Most explained sample index: {most_explained_idx}")
print(f"Total SHAP impact: {shap_total_impact[most_explained_idx]:.4f}")
assert isinstance(most_explained_idx, (int, np.integer)), "Should be an integer index"
print("Pass!")

In [ ]:
# Exercise 3: The SHAP base value for the RF model is the mean predicted probability
# on the training set. Verify this.

base_val = explainer.expected_value[1]
train_mean_prob = rf.predict_proba(X_train)[:, ___].mean()  # hint: column for class 1

print(f"SHAP base value:        {base_val:.4f}")
print(f"Mean train probability: {train_mean_prob:.4f}")
assert abs(base_val - train_mean_prob) < ___, "They should be very close"  # hint: use 0.01
print("Pass!")

In [ ]:
# Exercise 4: Create a DataFrame of SHAP values with feature names as columns
# for the test set (using shap_vals_default)

shap_df = pd.DataFrame(___, columns=___)  # hint: shap_vals_default, FEATURES

print(shap_df.shape)
print(shap_df.head(2))
assert shap_df.shape == (len(X_test), len(FEATURES)), "Shape should be (n_test, n_features)"
print("Pass!")

In [ ]:
# Exercise 5: Which customers in the test set have a SHAP value for 'credit_score'
# more negative than -0.1? (These are people where credit score strongly reduces their risk)

shap_df_full = pd.DataFrame(shap_vals_default, columns=FEATURES, index=X_test.index)
safe_credit = shap_df_full[shap_df_full['___'] < ___]  # fill in feature name and threshold

print(f"Customers with credit_score SHAP < -0.1: {len(safe_credit)}")
assert len(safe_credit) >= 0, "Should be a non-negative count"
print(f"Their avg credit score: {X_test.loc[safe_credit.index, 'credit_score'].mean():.0f}")
print("Pass!")

In [ ]:
# Exercise 6: For the customer segmentation model,
# which feature is most important for identifying Segment 0?

seg_mean_shap = [np.abs(seg_shap_vals[i]).mean(axis=0) for i in range(3)]
seg0_top_feature = seg_features[seg_mean_shap[___].argmax()]  # hint: segment index

print(f"Most important feature for Segment 0: {seg0_top_feature}")
assert seg0_top_feature in seg_features
print("Pass!")

In [ ]:
# Exercise 7: Plot a dependence plot for 'credit_score' SHAP values vs credit_score values
# A dependence plot shows how SHAP value changes as the feature value changes

shap.dependence_plot(
    '___',        # feature name
    ___,          # shap values array (shap_vals_default)
    X_test,
    feature_names=FEATURES,
    show=True
)

## Exercise Solutions

<details><summary>Click to reveal all solutions</summary>

**Exercise 1:**
```python
xgb_shap_abs_mean = np.abs(xgb_shap_vals).mean(axis=0)
most_important_feature = FEATURES[xgb_shap_abs_mean.argmax()]
```

**Exercise 2:**
```python
shap_total_impact = np.abs(shap_vals_default).sum(axis=1)
most_explained_idx = shap_total_impact.argmax()
```

**Exercise 3:**
```python
train_mean_prob = rf.predict_proba(X_train)[:, 1].mean()
assert abs(base_val - train_mean_prob) < 0.01
```

**Exercise 4:**
```python
shap_df = pd.DataFrame(shap_vals_default, columns=FEATURES)
```

**Exercise 5:**
```python
safe_credit = shap_df_full[shap_df_full['credit_score'] < -0.1]
```

**Exercise 6:**
```python
seg0_top_feature = seg_features[seg_mean_shap[0].argmax()]
```

**Exercise 7:**
```python
shap.dependence_plot('credit_score', shap_vals_default, X_test, feature_names=FEATURES, show=True)
```

</details>

## Cumulative Review — Days 1 through 9

Mixed exercises covering Pandas, NumPy, Data Cleaning, Faker, PyTorch basics, Train/Test splits, LogReg, Trees, Ensembles, and Classification Metrics.

In [ ]:
# Review 1 (Day 1 — Pandas): Group df by 'default' and compute mean income and credit_score
review1 = df.groupby('___')[['income', 'credit_score']].___()
print(review1)
assert review1.shape == (2, 2), "Should be 2 rows (0 and 1), 2 columns"
print("Pass!")

In [ ]:
# Review 2 (Day 2 — NumPy): Using broadcasting, normalize income to [0, 1] range
income_arr = df['income'].values
income_norm = (income_arr - income_arr.___()) / (income_arr.___() - income_arr.___())
assert income_norm.min() >= 0 and income_norm.max() <= 1, "Should be in [0,1]"
print(f"Min: {income_norm.min():.3f}, Max: {income_norm.max():.3f}")
print("Pass!")

In [ ]:
# Review 3 (Day 3 — Data Cleaning): Introduce some NaN values and then fill them
df_dirty = df.copy()
df_dirty.loc[np.random.choice(df_dirty.index, 50, replace=False), 'income'] = np.nan

print(f"NaN count before: {df_dirty['income'].isna().sum()}")
df_clean = df_dirty.copy()
df_clean['income'] = df_clean['income'].fillna(df_clean['income'].___())  # fill with median
print(f"NaN count after: {df_clean['income'].isna().sum()}")
assert df_clean['income'].isna().sum() == ___  # what should this be?
print("Pass!")

In [ ]:
# Review 4 (Day 6 — Pipelines): Build a quick sklearn Pipeline with StandardScaler + RF
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ('scaler', ___()),          # StandardScaler
    ('clf', RandomForestClassifier(n_estimators=50, random_state=0))
])
pipe.fit(X_train, y_train)
pipe_auc = roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1])
print(f"Pipeline AUC: {pipe_auc:.3f}")
assert pipe_auc > 0.5, "AUC should be above chance"
print("Pass!")

In [ ]:
# Review 5 (Day 7 — Trees): Train a Decision Tree and find its max depth used
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
print(f"Tree depth: {dt.tree_.max_depth}")
assert dt.tree_.max_depth <= ___, "Should not exceed max_depth=5"
print("Pass!")

In [ ]:
# Review 6 (Day 8 — Ensembles): Compare RF vs GradientBoosting AUC on the credit data
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)

rf_auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])
gb_auc = roc_auc_score(y_test, gb.predict_proba(X_test)[:, 1])
print(f"RF AUC:  {rf_auc:.3f}")
print(f"GB AUC:  {gb_auc:.3f}")
better = 'RF' if rf_auc > gb_auc else 'GB'
print(f"Winner: {better}")
assert rf_auc > 0.5 and gb_auc > 0.5
print("Pass!")

In [ ]:
# Review 7 (Day 9 — Metrics): Print a full classification report for XGBoost predictions
y_pred_xgb = xgb_model.predict(X_test)
print(classification_report(y_test, y_pred_xgb, target_names=['No Default', 'Default']))

# What is the recall for 'Default' class?
from sklearn.metrics import recall_score
default_recall = recall_score(y_test, y_pred_xgb)
print(f"Default recall: {default_recall:.3f}")
assert 0 <= default_recall <= ___  # fill in the max possible recall
print("Pass!")

In [ ]:
# Review 8 (Day 9 — ROC-AUC): Plot the ROC curve for all 3 models
from sklearn.metrics import roc_curve

fig, ax = plt.subplots(figsize=(7, 5))
for name, model in [('Random Forest', rf), ('XGBoost', xgb_model), ('Gradient Boosting', gb)]:
    probs = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")

ax.plot([0, 1], [0, 1], 'k--', label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Credit Risk Models')
ax.legend()
plt.tight_layout()
plt.show()

## Cumulative Review Solutions

<details><summary>Click to reveal</summary>

**Review 1:** `df.groupby('default')[['income', 'credit_score']].mean()`

**Review 2:** `income_arr.min()`, `income_arr.max()`, `income_arr.min()`

**Review 3:** `df_clean['income'].fillna(df_clean['income'].median())`, assert == 0

**Review 4:** `('scaler', StandardScaler())`

**Review 5:** `assert dt.tree_.max_depth <= 5`

**Review 6:** Whichever model has higher AUC wins (usually GB on this data)

**Review 7:** `assert 0 <= default_recall <= 1`

**Review 8:** No blanks — just run and inspect the chart

</details>

In [ ]:
# Cheat Sheet — SHAP Quick Reference
print("""
SHAP CHEAT SHEET
================

KEY CONCEPT
  prediction = base_value + sum(shap_values_i)
  - base_value = average model output on training data
  - shap_values_i = feature i's contribution to THIS prediction

SETUP
  import shap
  explainer = shap.TreeExplainer(model)
  shap_values = explainer.shap_values(X_test)

  # Binary classification -> shap_values is a list: [class0_vals, class1_vals]
  sv = shap_values[1]   # use class 1

  # Regression or multiclass XGBoost -> single array or list of arrays

PLOTS
  # Global importance (beeswarm)
  shap.summary_plot(sv, X_test)

  # Global importance (bar)
  shap.summary_plot(sv, X_test, plot_type='bar')

  # Single prediction breakdown
  shap.force_plot(explainer.expected_value[1], sv[i], X_test.iloc[i], matplotlib=True)
  shap.waterfall_plot(shap.Explanation(values=sv[i], base_values=..., data=...))

  # Feature effect vs value
  shap.dependence_plot('feature_name', sv, X_test)

GOTCHAS
  - shap_values is a LIST for binary RF classifiers (use [1])
  - SHAP values are in log-odds space for XGBoost/LogReg (not probabilities)
  - TreeExplainer: exact and fast for tree models
  - KernelExplainer: slow but works on any model
  - Always use X_test (not X_train) for SHAP plots unless you want training insights
""")


---
## Phase 2 Complete! Great work.

You've finished all 5 days of Classical ML:

- Day 6: Train/Test splits and Pipelines
- Day 7: Logistic Regression and Decision Trees
- Day 8: Ensembles and XGBoost
- Day 9: Classification Metrics
- Day 10: SHAP and Mini Projects ✓

---

**Next up: Day 11 — Phase 3 NLP — TextPreprocessingAndTFIDF**

Tokenization, stopwords, stemming, CountVectorizer, TF-IDF. The bridge into NLP!